In [1]:
import os
import pandas as pd
import numpy as np
import scipy
import math
import plotnine
import matplotlib.pyplot as plt
import warnings
from scipy.integrate import solve_ivp
from datetime import datetime
from datetime import timedelta

In [3]:
all_scalings= pd.read_csv("../data/scaling_relationships.csv")

half_sats_p = pd.read_csv("../../2026_08_29_complete_workflow/output_estimation_logN_linP_20260918/half_sats/phosphorus_half_sats.csv")
half_sats_p = half_sats_p.loc[half_sats_p.exponential_p_val < 0.15,:]
half_sats_n = pd.read_csv("../../2026_08_29_complete_workflow/output_estimation_logN_linP_20260918/half_sats/nitrogen_half_sats.csv")
half_sats_n = half_sats_n.loc[half_sats_n.exponential_p_val < 0.15,:]

def get_half_sat(Taxon,nutrient="P"):
    half_sats=half_sats_n
    if nutrient=="P":
        half_sats=half_sats_p.copy(deep=True)
        half_sats.estimated_parameter=half_sats.estimated_parameter
        if Taxon=="Phaeocystales":
            half_sats.loc[half_sats.Taxon==Taxon,
                               "estimated_parameter"] = 1
    if Taxon not in list(half_sats.Taxon):
        return "not_found"
    return float(half_sats.loc[half_sats.Taxon==Taxon,
                               "estimated_parameter"])

temp_relation=pd.read_csv("../../2026_08_29_complete_workflow/output_estimation_logN_linP_20260916/temp_relation/temperature_mean_std.csv")
def get_temp_params(Taxon,param="mean"):
    if param=="mean":
        return float(temp_relation.loc[temp_relation.Order==Taxon,"mean_temp"])
    if param=="std":
        return float(temp_relation.loc[temp_relation.Order==Taxon,"std_temp"])

In [5]:
all_scalings

,Unnamed: 0,taxon,diameter,pft,parameter_a_gr,parameter_a_gr_denom,parameter_b_gr,parameter_a_no3,parameter_b_no3,parameter_a_no2,parameter_b_no2,parameter_a_nh4,parameter_b_nh4,parameter_a_p,parameter_b_p,darwin_from_class,number_group,max_growth_rate,half_sat_n,half_sat_p
0,0,Mamiellales,1.5,PicoEuk,0.000013,86400.0,0.28,0.17,0.27,0.17,0.27,0.085,0.27,0.026,0.27,PicoEuk,3,1.301610,0.495624,0.030321
1,1,Leptocylindrales,15.0,Diatom,0.000029,86400.0,-0.10,0.17,0.27,0.17,0.27,0.085,0.27,0.026,0.27,Diatom,7,1.182059,3.200018,0.195766
2,2,Cryptomonadales,15.0,PicoEuk,0.000013,86400.0,0.28,0.17,0.27,0.17,0.27,0.085,0.27,0.026,0.27,PicoEuk,3,9.004944,3.200018,0.195766
3,3,Cymatosirales,5.0,Diatom,0.000029,86400.0,-0.10,0.17,0.27,0.17,0.27,0.085,0.27,0.026,0.27,Diatom,7,1.643522,1.314269,0.080402
4,4,Prymnesiales,5.0,PicoEuk,0.000013,86400.0,0.28,0.17,0.27,0.17,0.27,0.085,0.27,0.026,0.27,PicoEuk,3,3.578484,1.314269,0.080402
5,5,Suessiales,10.0,Dino,0.000015,86400.0,-0.10,0.17,0.27,0.17,0.27,0.085,0.27,0.026,0.27,Dino,8,0.714503,2.304191,0.140962
6,6,Hemiaulales,50.0,Diatom,0.000029,86400.0,-0.10,0.17,0.27,0.17,0.27,0.085,0.27,0.026,0.27,Diatom,7,0.823712,8.485633,0.519121
7,7,Chaetocerotophycidae,25.0,Diatom,0.000029,86400.0,-0.10,0.17,0.27,0.17,0.27,0.085,0.27,0.026,0.27,Diatom,7,1.014109,4.840051,0.296097
8,8,Rhizosoleniales,100.0,Diatom,0.000029,86400.0,-0.10,0.17,0.27,0.17,0.27,0.085,0.27,0.026,0.27,Diatom,7,0.669062,14.877109,0.910129
9,9,Gymnodiniales,10.0,Dino,0.000015,86400.0,-0.10,0.17,0.27,0.17,0.27,0.085,0.27,0.026,0.27,Dino,8,0.714503,2.304191,0.140962


In [4]:
organisms_for_model = list(set(half_sats_p.Taxon).union(set(half_sats_n.Taxon)))

In [ ]:
def get_max_gr(taxon):
    return float(all_scalings.loc[all_scalings.taxon==taxon,"max_growth_rate"])

### parameters from the 2026 exercise
consistent_k_light = 500
consistent_T_opt = 8
consistent_std_T = 5
consistent_max_gr = 1
consistent_half_sat_silicon = 1
consistent_half_sat_metab = 0.5
consistent_half_sat_nitr = 1
consistent_half_sat_phos = 0.1

params_frame={}
for organism in organisms_for_model:
    half_sat_nitr = get_half_sat(organism,"N")
    half_sat_phos = get_half_sat(organism,"P")
    if half_sat_nitr == "not_found":
        half_sat_nitr = consistent_half_sat_nitr
    if half_sat_phos == "not_found":
        half_sat_phos = consistent_half_sat_phos
    max_gr=consistent_max_gr # change to maximum weighted biomass
    
    current_org_dict={"max_gr":get_max_gr(organism),
                      "half_sat_nitr":half_sat_nitr,
                      "half_sat_phos":half_sat_phos,
                      "T_opt":get_temp_params(organism,"mean"),
                      "std_T":get_temp_params(organism,"std")}
    params_frame[organism]=current_org_dict